In [2]:
#import libraries
import pandas as pd
import numpy as np
import torch

#load dataset

df = pd.read_csv("/home/vmasc/Documents/ECE883_Digital_Image_processing/Final_project/dataset_IQ/master_iq_dataset.csv")
df["label"].value_counts()

label
0    2000000
1    2000000
Name: count, dtype: int64

In [3]:
# Shuffle rows
print(df.head())
print(df.shape)

#create sample id row
rows_per_sample = 2000
df["sample_id"] = np.arange(len(df)) // rows_per_sample
print(df[["sample_id", "label"]].head(10))
print(df["sample_id"].nunique())



#CONVERT PATHMODEL LOSS TO NUMERIC DATA 
df["pathLossModel"] = df["pathLossModel"].astype("category").cat.codes


   I_component  Q_component  label   SNR_dB  SIR_dB  numRx  numUE      UE1_x  \
0     0.000006    -0.000010      0  15.2566     NaN      4      3 -1415.2664   
1     0.000013    -0.000013      0  15.2566     NaN      4      3 -1415.2664   
2    -0.000008    -0.000004      0  15.2566     NaN      4      3 -1415.2664   
3     0.000016    -0.000011      0  15.2566     NaN      4      3 -1415.2664   
4     0.000017     0.000003      0  15.2566     NaN      4      3 -1415.2664   

      UE1_y  UE1_z  ...      UE3_x    UE3_y  UE3_z  gNB_x  gNB_y  gNB_z  \
0  103.0068    0.0  ... -1117.5836 -686.425    0.0    0.0    0.0    0.0   
1  103.0068    0.0  ... -1117.5836 -686.425    0.0    0.0    0.0    0.0   
2  103.0068    0.0  ... -1117.5836 -686.425    0.0    0.0    0.0    0.0   
3  103.0068    0.0  ... -1117.5836 -686.425    0.0    0.0    0.0    0.0   
4  103.0068    0.0  ... -1117.5836 -686.425    0.0    0.0    0.0    0.0   

   radar_x  radar_y  radar_z  pathLossModel  
0   3000.0    400.0   

In [4]:
from torch_geometric.data import Data

df["magnitude"] = np.sqrt(df["I_component"]**2 + df["Q_component"]**2)
df["phase"] = np.arctan2(df["Q_component"], df["I_component"])

feature_cols = ["I_component", "Q_component", "magnitude", "phase"]
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[feature_cols] = scaler.fit_transform(df[feature_cols])

def build_edges(num_nodes):
    edges = []

    for i in range(num_nodes):
        for offset in range(1, 5):
            if i - offset >= 0:
                edges.append([i, i - offset])
            if i + offset < num_nodes:
                edges.append([i, i + offset])

    return torch.tensor(edges, dtype=torch.long).t().contiguous()

def build_graph_from_group(group, feature_cols):
    group = group.reset_index(drop=True)

    x_np = group[feature_cols].to_numpy(dtype=np.float32)
    x = torch.from_numpy(x_np)

    edge_index = build_edges(len(group))
    y = torch.tensor([int(group["label"].iloc[0])], dtype=torch.long)

    return Data(x=x, edge_index=edge_index, y=y)

/home/vmasc/Documents/ECE883_Digital_Image_processing/Final_project/gnn_env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
graphs = []

for sample_id, group in df.groupby("sample_id"):
    if len(group) != rows_per_sample:
        continue
    graph = build_graph_from_group(group, feature_cols)
    graphs.append(graph)

print(graphs[0])
print(graphs[0].x.shape)
print(graphs[0].edge_index.shape)
print(graphs[0].y)

Data(x=[2000, 4], edge_index=[2, 15980], y=[1])
torch.Size([2000, 4])
torch.Size([2, 15980])
tensor([0])


In [6]:
print(df[feature_cols].dtypes)


I_component    float64
Q_component    float64
magnitude      float64
phase          float64
dtype: object


In [7]:
mapping = {
    "FSPL": 0,
    "LogDistance": 1,
    "UMa": 2
}

df["pathLossModel"] = df["pathLossModel"].map(mapping)

In [8]:
print("x shape:", graphs[0].x.shape)
print("edge_index shape:", graphs[0].edge_index.shape)
print("y:", graphs[0].y)

x shape: torch.Size([2000, 4])
edge_index shape: torch.Size([2, 15980])
y: tensor([0])


In [9]:
from sklearn.model_selection import train_test_split

labels = [g.y.item() for g in graphs]

train_graphs, test_graphs = train_test_split(
    graphs,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

print("Train graphs:", len(train_graphs))
print("Test graphs:", len(test_graphs))

Train graphs: 1600
Test graphs: 400


In [10]:
from torch_geometric.loader import DataLoader

train_loader = DataLoader(train_graphs, batch_size=128, shuffle=True)
test_loader = DataLoader(test_graphs, batch_size=128, shuffle=False)

In [15]:
import model ,importlib
importlib.reload(model)
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.GINClassifier(
    in_dim=4,
    hidden=64,
    num_layers=4,
    num_classes=2,
    dropout=0.2
)
model = model.to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

In [16]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    total_correct = 0
    total_graphs = 0

    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        out = model(batch)                  # shape: [batch_size, num_classes]
        loss = F.cross_entropy(out, batch.y)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = out.argmax(dim=1)
        total_correct += (preds == batch.y).sum().item()
        total_graphs += batch.y.size(0)

    avg_loss = total_loss / len(loader)
    avg_acc = total_correct / total_graphs
    return avg_loss, avg_acc

In [17]:
def evaluate(model, loader, device):
    model.eval()
    total_loss = 0
    total_correct = 0
    total_graphs = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)

            out = model(batch)
            loss = F.cross_entropy(out, batch.y)

            total_loss += loss.item()
            preds = out.argmax(dim=1)

            total_correct += (preds == batch.y).sum().item()
            total_graphs += batch.y.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(batch.y.cpu().numpy())

    avg_loss = total_loss / len(loader)
    avg_acc = total_correct / total_graphs
    return avg_loss, avg_acc, all_preds, all_labels

In [18]:
num_epochs = 20

for epoch in range(1, num_epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    test_loss, test_acc, test_preds, test_labels = evaluate(model, test_loader, device)

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}"
    )

Epoch 01 | Train Loss: 0.6973 | Train Acc: 0.5156 | Test Loss: 0.7005 | Test Acc: 0.4775
Epoch 02 | Train Loss: 0.6931 | Train Acc: 0.5081 | Test Loss: 0.6901 | Test Acc: 0.4925
Epoch 03 | Train Loss: 0.6917 | Train Acc: 0.5188 | Test Loss: 0.6927 | Test Acc: 0.4600
Epoch 04 | Train Loss: 0.6917 | Train Acc: 0.5156 | Test Loss: 0.6950 | Test Acc: 0.4875
Epoch 05 | Train Loss: 0.6920 | Train Acc: 0.5044 | Test Loss: 0.6928 | Test Acc: 0.4875
Epoch 06 | Train Loss: 0.6894 | Train Acc: 0.5363 | Test Loss: 0.6974 | Test Acc: 0.4675
Epoch 07 | Train Loss: 0.6917 | Train Acc: 0.5112 | Test Loss: 0.6932 | Test Acc: 0.4825
Epoch 08 | Train Loss: 0.6911 | Train Acc: 0.5094 | Test Loss: 0.6937 | Test Acc: 0.5025
Epoch 09 | Train Loss: 0.6872 | Train Acc: 0.5325 | Test Loss: 0.6993 | Test Acc: 0.4925
Epoch 10 | Train Loss: 0.6878 | Train Acc: 0.5369 | Test Loss: 0.6966 | Test Acc: 0.5125
Epoch 11 | Train Loss: 0.6916 | Train Acc: 0.5112 | Test Loss: 0.6920 | Test Acc: 0.5375
Epoch 12 | Train Loss

In [20]:
model.eval()  # switch to evaluation mode

with torch.no_grad():  # disable gradient computation
    outputs = model(test_loader)
    predictions = torch.argmax(outputs, dim=1)

AttributeError: 'DataLoader' object has no attribute 'x'